# 03: Injury / Workload Risk Score
**Goal:** build a simple, explainable risk score from injury frequency, recency, and workload. This does NOT need to be a black-box ML model -- a well-justified composite score is more useful (and more credible) to a recruitment audience than something you can't explain in plain language.

**Known data limitation going in:** injury counts are capped at 60 for some players due to a pagination limit in the source dataset -- see DEBUGGING_LOG.md #10. This means the frequency component will treat several different 'very high injury' players as tied at the max, which is a reasonable approximation, not a precise ranking among them.

## Setup

In [1]:
import duckdb
import pandas as pd
import numpy as np

## Load injury/workload data
Connecting directly to the DuckDB warehouse dbt writes to.

In [2]:
con = duckdb.connect('../data/warehouse.duckdb')
injury_workload = con.sql('SELECT * FROM main.player_injury_workload').df()
injury_workload['most_recent_injury_date'] = pd.to_datetime(
    injury_workload['most_recent_injury_date']
)
injury_workload.head()

,player_id,canonical_name,injury_count,total_days_missed,most_recent_injury_date,total_90s_played,total_matches
0,137150,Alex Wynter,1,193.0,2022-11-19,0.0,0.0
1,170322,Jonathan Williams,5,340.0,2013-09-09,0.0,0.0
2,104962,Filip Orsula,1,86.0,2019-04-05,0.0,0.0
3,520651,Jensen Weir,1,167.0,2022-01-14,0.0,0.0
4,3271,Luke Young,2,79.0,2012-02-04,0.0,0.0


In [3]:
n_with_zero_days = ((injury_workload['injury_count'] > 0) & (injury_workload['total_days_missed'] == 0)).sum()
n_with_injuries = (injury_workload['injury_count'] > 0).sum()
print(f'{n_with_zero_days} / {n_with_injuries} players with injuries show 0 total days missed')

1 / 1176 players with injuries show 0 total days missed


In [4]:
con2 = duckdb.connect('../data/warehouse.duckdb')
print(con2.sql("SELECT \"Days\", \"Games missed\" FROM raw.injuries LIMIT 10").df())

       Days Games missed
0   90 days           11
1   22 days            4
2   22 days            2
3    7 days            1
4  196 days           24
5  122 days           16
6   46 days            8
7   31 days            4
8   21 days            5
9   66 days            9


In [ ]:
con.close()

In [ ]:
con2.close()

## Handle players with zero injuries
A player with `injury_count == 0` has `most_recent_injury_date == NaT` (no injury to date from). This isn't missing data -- it's a genuinely different case that needs explicit handling before the recency component, not a silent default.

In [5]:
n_zero_injury = (injury_workload['injury_count'] == 0).sum()
print(f'{n_zero_injury} / {len(injury_workload)} players have zero recorded injuries')

# Confirm these are exactly the players with a null date -- if not, that's a data issue to investigate
mismatch = injury_workload[
    (injury_workload['injury_count'] == 0) != injury_workload['most_recent_injury_date'].isna()
]
print(f'Mismatches between zero-injury and null-date: {len(mismatch)} (should be 0)')

783 / 1959 players have zero recorded injuries
Mismatches between zero-injury and null-date: 0 (should be 0)


## Design the score
Three components, each normalized to 0-1 before combining so no single component dominates just because of its raw scale:
- **Frequency** -- injury_count relative to the rest of the league
- **Recency** -- how recently the last injury occurred (more recent = higher risk). Players with zero injuries get the lowest possible recency score (0), not a missing value -- no injury history is genuinely the lowest-risk case here, not an unknown one.
- **Load** -- total_90s_played relative to the rest of the league (high load = higher risk of future injury)

**Weighting choice (documented explicitly, since this is the kind of assumption worth being able to explain in the presentation):** frequency weighted highest (0.4) since repeated injuries are the strongest direct signal; recency and load weighted equally (0.3 each) as secondary signals. This is a judgment call, not a derived result -- stated plainly rather than presented as objectively optimal.

In [6]:
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

injury_workload['frequency_score'] = normalize(injury_workload['injury_count'])

# Recency: days since most recent injury, inverted so recent = high risk.
# Zero-injury players get recency_score = 0 explicitly (lowest risk),
# not from normalizing a missing value.
today = pd.Timestamp.today()
days_since = (today - injury_workload['most_recent_injury_date']).dt.days
recency_score = 1 - normalize(days_since)
injury_workload['recency_score'] = recency_score.fillna(0)

injury_workload['load_score'] = normalize(injury_workload['total_90s_played'])

In [7]:
weights = {'frequency': 0.4, 'recency': 0.3, 'load': 0.3}

injury_workload['risk_score'] = (
    weights['frequency'] * injury_workload['frequency_score'] +
    weights['recency'] * injury_workload['recency_score'] +
    weights['load'] * injury_workload['load_score']
)

injury_workload[['player_id', 'canonical_name', 'injury_count',
                  'total_90s_played', 'risk_score']].sort_values(
    'risk_score', ascending=False).head(20)

,player_id,canonical_name,injury_count,total_90s_played,risk_score
212,92571,Aaron Cresswell,60,5475.588889,0.815522
589,148368,Divock Origi,60,436.144444,0.694165
303,135343,Fabian Schär,60,4294.422222,0.693314
669,502821,Riccardo Calafiori,60,0.000000,0.679811
511,55125,Chris Löwe,60,1068.633333,0.674158
1126,37304,Shane Long,60,1342.288889,0.667916
90,13520,Phil Jagielka,60,674.066667,0.632373
1117,403898,Jordan Beyer,60,205.666667,0.631461
1037,5658,Petr Cech,60,0.000000,0.624914
536,130164,Jordan Pickford,4,9314.466667,0.606784


In [8]:
injury_workload['hit_injury_cap'] = injury_workload['injury_count'] == 60

n_capped_in_top20 = injury_workload.sort_values('risk_score', ascending=False).head(20)['hit_injury_cap'].sum()
print(f'{n_capped_in_top20} / 20 highest-risk players hit the injury-count data cap')

12 / 20 highest-risk players hit the injury-count data cap


## Sanity-check the top and bottom of the list
Do the highest-risk players match what you'd expect from football knowledge (recent long-term injuries, heavy fixture load)? Do the lowest-risk players make sense too (young players, low minutes, no injury history)? If the ranking looks wrong, revisit the weights before moving on -- this is a judgment call worth getting right, and worth actually looking at rather than trusting blindly.

In [9]:
print('=== Highest risk ===')
print(injury_workload[['canonical_name', 'injury_count', 'total_days_missed',
                        'total_90s_played', 'risk_score']]
      .sort_values('risk_score', ascending=False).head(10).to_string(index=False))

print('\n=== Lowest risk ===')
print(injury_workload[['canonical_name', 'injury_count', 'total_days_missed',
                        'total_90s_played', 'risk_score']]
      .sort_values('risk_score', ascending=True).head(10).to_string(index=False))

=== Highest risk ===
    canonical_name  injury_count  total_days_missed  total_90s_played  risk_score
   Aaron Cresswell            60              904.0       5475.588889    0.815522
      Divock Origi            60             1028.0        436.144444    0.694165
      Fabian Schär            60             1001.0       4294.422222    0.693314
Riccardo Calafiori            60             2268.0          0.000000    0.679811
        Chris Löwe            60             3676.0       1068.633333    0.674158
        Shane Long            60             1100.0       1342.288889    0.667916
     Phil Jagielka            60             2696.0        674.066667    0.632373
      Jordan Beyer            60             1304.0        205.666667    0.631461
         Petr Cech            60             1640.0          0.000000    0.624914
   Jordan Pickford             4              116.0       9314.466667    0.606784

=== Lowest risk ===
canonical_name  injury_count  total_days_missed  total_9

## Save for notebook 04

In [10]:
import os
os.makedirs('../data/processed', exist_ok=True)
injury_workload.to_csv('../data/processed/risk_scores.csv', index=False)
print(f'Saved {len(injury_workload)} players to risk_scores.csv')

Saved 1959 players to risk_scores.csv
